In [ ]:
# =============================================================================
# PixelNet for Smoke and Fire Semantic Segmentation
# Complete Implementation - Google Colab Ready
# Architecture: VGG16 Backbone + Hypercolumn + MLP Classifier
# =============================================================================


# =============================================================================
# SNIPPET 1: Environment Setup
# =============================================================================

# Install required libraries (run in Colab)
# !pip install torch torchvision tqdm matplotlib kaggle

import os
import sys
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import functional as TF

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm import tqdm

# ----- Device Configuration -----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ----- Reproducibility -----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----- Global Constants -----
IMAGE_SIZE = (224, 224)       # Input resolution for VGG16
NUM_CLASSES = 2               # 0 = background, 1 = fire/smoke
NUM_SAMPLED_PIXELS = 2000     # Pixels sampled per image during training
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
NUM_EPOCHS = 20

print("Environment setup complete.")


# =============================================================================
# SNIPPET 2: Kaggle Dataset Download
# =============================================================================

def setup_kaggle_credentials(kaggle_json_path: str = "kaggle.json"):
    """
    Configure Kaggle API credentials from the uploaded kaggle.json file.
    Copies the file to ~/.kaggle/ and sets correct permissions.
    """
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    dest = kaggle_dir / "kaggle.json"
    shutil.copy(kaggle_json_path, dest)
    os.chmod(dest, 0o600)  # Required permission for Kaggle API

    print("Kaggle credentials configured successfully.")


def download_and_extract_dataset(
    dataset_slug: str = "diversisai/fire-and-smoke",
    download_dir: str = "data/raw",
    extract_dir: str = "data/extracted"
):
    """
    Download a Kaggle dataset and extract its contents.

    Args:
        dataset_slug : Kaggle dataset identifier (username/dataset-name)
        download_dir : Directory to save the downloaded ZIP
        extract_dir  : Directory to extract dataset files into
    """
    os.makedirs(download_dir, exist_ok=True)
    os.makedirs(extract_dir, exist_ok=True)

    # Download via Kaggle CLI
    print(f"Downloading dataset: {dataset_slug}")
    os.system(f"kaggle datasets download -d {dataset_slug} -p {download_dir}")

    # Find and extract the downloaded ZIP file
    zip_files = list(Path(download_dir).glob("*.zip"))
    if not zip_files:
        raise FileNotFoundError("No ZIP file found after download. Check dataset slug.")

    zip_path = zip_files[0]
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    print(f"Dataset extracted to: {extract_dir}")
    return extract_dir


def organize_dataset(
    extracted_dir: str = "data/extracted",
    output_dir: str = "dataset"
):
    """
    Organize extracted files into a clean structure:
        dataset/
            images/   <- RGB input images
            masks/    <- Binary segmentation masks

    NOTE: Adjust glob patterns below to match the actual folder structure
    of the downloaded Kaggle dataset.
    """
    images_out = Path(output_dir) / "images"
    masks_out  = Path(output_dir) / "masks"
    images_out.mkdir(parents=True, exist_ok=True)
    masks_out.mkdir(parents=True, exist_ok=True)

    extracted = Path(extracted_dir)

    # --- Collect image files ---
    image_files = (
        list(extracted.rglob("*.jpg")) +
        list(extracted.rglob("*.jpeg")) +
        list(extracted.rglob("*.png"))
    )

    # Separate images and masks by filename convention
    # Convention: mask files contain "_mask" or are in a "masks/" subfolder
    for f in image_files:
        if "mask" in f.stem.lower() or "masks" in str(f.parent).lower():
            shutil.copy(f, masks_out / f.name)
        else:
            shutil.copy(f, images_out / f.name)

    img_count  = len(list(images_out.glob("*")))
    mask_count = len(list(masks_out.glob("*")))
    print(f"Organized dataset — Images: {img_count}, Masks: {mask_count}")
    return str(images_out), str(masks_out)


# ----- How to use (run these lines in Colab) -----
# from google.colab import files
# uploaded = files.upload()                          # Upload kaggle.json
# setup_kaggle_credentials("kaggle.json")
# extract_dir = download_and_extract_dataset()
# images_dir, masks_dir = organize_dataset(extract_dir)


# =============================================================================
# SNIPPET 3: Dataset Loader
# =============================================================================

class FireSmokeDataset(Dataset):
    """
    PyTorch Dataset for Fire and Smoke Semantic Segmentation.

    Each sample consists of:
        image : Normalized RGB tensor  [3, H, W]
        mask  : Binary mask tensor     [H, W]  (0=background, 1=fire/smoke)

    The mask conversion logic thresholds pixel intensity:
        - If the mask is grayscale: pixels > 127 → fire/smoke (class 1)
        - If the mask is RGB: non-black pixels → fire/smoke (class 1)
    """

    # ImageNet normalization stats (required for VGG16 pretrained weights)
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(
        self,
        images_dir: str,
        masks_dir: str,
        image_size: tuple = (224, 224),
        augment: bool = False
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.image_size = image_size
        self.augment    = augment

        # Match each image to its corresponding mask by stem (filename without extension)
        self.samples = self._pair_images_and_masks()
        assert len(self.samples) > 0, "No image-mask pairs found. Check directory paths."

        # Image transform: resize + tensor + normalize
        self.image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.MEAN, std=self.STD),
        ])

        print(f"Dataset loaded: {len(self.samples)} image-mask pairs")

    def _pair_images_and_masks(self):
        """
        Pair image files with their corresponding mask files using stem matching.
        Supports: .jpg, .jpeg, .png extensions.
        """
        image_files = {
            f.stem: f
            for f in self.images_dir.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
        }
        mask_files = {
            f.stem.replace("_mask", ""): f
            for f in self.masks_dir.iterdir()
            if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
        }

        # Keep only pairs where both image and mask exist
        paired_stems = set(image_files.keys()) & set(mask_files.keys())
        return [(image_files[s], mask_files[s]) for s in sorted(paired_stems)]

    def _load_binary_mask(self, mask_path: Path) -> Image.Image:
        """
        Load mask and convert to a binary PIL image (0 or 255).
        Handles both grayscale and RGB masks.
        """
        mask = Image.open(mask_path)

        if mask.mode == "L":                         # Grayscale mask
            mask_array = np.array(mask)
            binary = (mask_array > 127).astype(np.uint8) * 255
        else:                                         # RGB mask
            mask_rgb = np.array(mask.convert("RGB"))
            # Non-black pixels are fire/smoke
            binary = (mask_rgb.sum(axis=2) > 30).astype(np.uint8) * 255

        return Image.fromarray(binary, mode="L")

    def _apply_augmentation(self, image: Image.Image, mask: Image.Image):
        """
        Apply synchronized random augmentations to image and mask.
        Both must receive identical spatial transforms.
        """
        # Random horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            mask  = TF.hflip(mask)

        # Random vertical flip
        if random.random() > 0.3:
            image = TF.vflip(image)
            mask  = TF.vflip(mask)

        # Random rotation (±15 degrees)
        angle = random.uniform(-15, 15)
        image = TF.rotate(image, angle)
        mask  = TF.rotate(mask, angle)

        return image, mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        # Load and prepare image
        image = Image.open(image_path).convert("RGB")

        # Load and binarize mask
        mask = self._load_binary_mask(mask_path)
        mask = mask.resize(self.image_size, Image.NEAREST)  # Preserve binary values

        # Optional augmentation (training only)
        if self.augment:
            image, mask = self._apply_augmentation(image, mask)

        # Apply image transforms (resize + normalize)
        image_tensor = self.image_transform(image)

        # Convert mask to long tensor with class labels {0, 1}
        mask_array  = np.array(mask)
        mask_tensor = torch.from_numpy((mask_array > 127).astype(np.int64))

        return image_tensor, mask_tensor


# =============================================================================
# SNIPPET 4: DataLoader
# =============================================================================

def build_dataloaders(
    images_dir: str,
    masks_dir: str,
    image_size: tuple = IMAGE_SIZE,
    batch_size: int = BATCH_SIZE,
    train_split: float = 0.8,
    num_workers: int = 2
):
    """
    Build train and validation DataLoaders from the dataset directory.

    Args:
        images_dir  : Path to images folder
        masks_dir   : Path to masks folder
        image_size  : (H, W) to resize inputs
        batch_size  : Samples per batch
        train_split : Fraction of data for training
        num_workers : Parallel data loading workers

    Returns:
        train_loader, val_loader
    """
    # Full dataset (no augmentation for now; we split first)
    full_dataset = FireSmokeDataset(images_dir, masks_dir, image_size, augment=False)

    n_total = len(full_dataset)
    n_train = int(n_total * train_split)
    n_val   = n_total - n_train

    # Split into train / val subsets
    train_set, val_set = torch.utils.data.random_split(
        full_dataset,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Enable augmentation only for the training subset
    train_set.dataset.augment = True

    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
    return train_loader, val_loader


# =============================================================================
# SNIPPET 5: CNN Backbone (VGG16 Multi-Scale Feature Extractor)
# =============================================================================

class VGG16Backbone(nn.Module):
    """
    Pretrained VGG16 backbone for multi-scale feature extraction.

    Extracts feature maps from four intermediate stages of VGG16:
        Stage 1 (relu1_2) : 64  channels — shallow, high-res edges
        Stage 2 (relu2_2) : 128 channels — textures and color blobs
        Stage 3 (relu3_3) : 256 channels — object parts
        Stage 4 (relu4_3) : 512 channels — high-level semantics

    These multi-scale features are the foundation of the Hypercolumn.
    """

    def __init__(self, pretrained: bool = True):
        super(VGG16Backbone, self).__init__()

        # Load pretrained VGG16
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        features = vgg.features  # Only the convolutional layers

        # Slice VGG16's features into 4 stages at specific layer indices
        # VGG16 layer index map:
        #   relu1_2 = index 4  | relu2_2 = index 9
        #   relu3_3 = index 16 | relu4_3 = index 23
        self.stage1 = nn.Sequential(*list(features.children())[:5])    # relu1_2
        self.stage2 = nn.Sequential(*list(features.children())[5:10])  # relu2_2
        self.stage3 = nn.Sequential(*list(features.children())[10:17]) # relu3_3
        self.stage4 = nn.Sequential(*list(features.children())[17:24]) # relu4_3

        # Freeze backbone to preserve pretrained weights during early training
        for param in self.parameters():
            param.requires_grad = False

    def unfreeze(self):
        """Optionally unfreeze backbone for fine-tuning."""
        for param in self.parameters():
            param.requires_grad = True

    def forward(self, x: torch.Tensor):
        """
        Args:
            x : Input image tensor [B, 3, H, W]

        Returns:
            Dictionary of feature maps at 4 scales
        """
        f1 = self.stage1(x)    # [B, 64,  H/2,  W/2]
        f2 = self.stage2(f1)   # [B, 128, H/4,  W/4]
        f3 = self.stage3(f2)   # [B, 256, H/8,  W/8]
        f4 = self.stage4(f3)   # [B, 512, H/16, W/16]

        return {"f1": f1, "f2": f2, "f3": f3, "f4": f4}


# =============================================================================
# SNIPPET 6: Hypercolumn Construction
# =============================================================================

class HypercolumnBuilder(nn.Module):
    """
    Constructs Hypercolumn representations by upsampling all feature maps
    to the original input resolution and concatenating them channel-wise.

    Hypercolumn at pixel (x, y) = concatenation of activations at that
    pixel position across ALL feature scales.

    Output channels = 64 + 128 + 256 + 512 = 960
    """

    def __init__(self):
        super(HypercolumnBuilder, self).__init__()

    def forward(self, feature_dict: dict, target_size: tuple) -> torch.Tensor:
        """
        Args:
            feature_dict : {"f1": tensor, "f2": tensor, "f3": tensor, "f4": tensor}
            target_size  : (H, W) — original input spatial dimensions

        Returns:
            hypercolumn : [B, 960, H, W] — concatenated multi-scale features
        """
        upsampled = []

        for key in ["f1", "f2", "f3", "f4"]:
            feat = feature_dict[key]

            # Bilinear upsample to match input resolution
            if feat.shape[-2:] != target_size:
                feat = F.interpolate(
                    feat,
                    size=target_size,
                    mode="bilinear",
                    align_corners=False
                )
            upsampled.append(feat)

        # Concatenate along channel dimension: [B, 960, H, W]
        hypercolumn = torch.cat(upsampled, dim=1)
        return hypercolumn


# =============================================================================
# SNIPPET 7: Pixel Sampling
# =============================================================================

def sample_pixels(
    hypercolumn: torch.Tensor,
    mask: torch.Tensor,
    num_pixels: int = NUM_SAMPLED_PIXELS
):
    """
    PixelNet's core training mechanism: instead of predicting ALL pixels,
    we randomly sample a subset and train only on those.

    This enables:
    - Efficient training (fewer forward passes per image)
    - Better class balance (can oversample fire/smoke pixels)
    - Compatibility with fully connected MLP classifiers

    Args:
        hypercolumn : [B, C, H, W] — full hypercolumn feature map
        mask        : [B, H, W]    — ground truth binary labels
        num_pixels  : Number of pixels to sample per batch image

    Returns:
        sampled_features : [B * num_pixels, C]
        sampled_labels   : [B * num_pixels]
    """
    B, C, H, W = hypercolumn.shape
    all_features = []
    all_labels   = []

    for b in range(B):
        hc     = hypercolumn[b]       # [C, H, W]
        lbl    = mask[b].view(-1)     # [H*W]

        # Reshape hypercolumn to [H*W, C] for indexing by pixel
        hc_flat = hc.view(C, -1).permute(1, 0)  # [H*W, C]

        total_pixels = H * W
        n_sample = min(num_pixels, total_pixels)

        # Stratified sampling: try to balance fire/smoke and background pixels
        fire_indices = (lbl == 1).nonzero(as_tuple=True)[0]
        bg_indices   = (lbl == 0).nonzero(as_tuple=True)[0]

        n_fire = min(n_sample // 2, len(fire_indices))
        n_bg   = n_sample - n_fire

        selected_fire = fire_indices[torch.randperm(len(fire_indices))[:n_fire]]
        selected_bg   = bg_indices[torch.randperm(len(bg_indices))[:n_bg]]

        selected = torch.cat([selected_fire, selected_bg])

        all_features.append(hc_flat[selected])   # [n_sample, C]
        all_labels.append(lbl[selected])          # [n_sample]

    sampled_features = torch.cat(all_features, dim=0)  # [B*n_sample, C]
    sampled_labels   = torch.cat(all_labels,   dim=0)  # [B*n_sample]

    return sampled_features, sampled_labels


# =============================================================================
# SNIPPET 8: MLP Classifier
# =============================================================================

class PixelMLP(nn.Module):
    """
    MLP (Multi-Layer Perceptron) classifier for per-pixel classification.

    Takes a hypercolumn feature vector for a single pixel and outputs
    class logits (background vs fire/smoke).

    Architecture:
        Linear(960 → 512) → BN → ReLU → Dropout
        Linear(512 → 256) → BN → ReLU → Dropout
        Linear(256 →   2)        [class logits]

    Input:  [N, 960]  — N sampled pixels with hypercolumn features
    Output: [N, 2]    — raw logits for {background, fire/smoke}
    """

    def __init__(
        self,
        input_dim: int = 960,       # 64 + 128 + 256 + 512
        hidden1: int = 512,
        hidden2: int = 256,
        num_classes: int = NUM_CLASSES,
        dropout_rate: float = 0.4
    ):
        super(PixelMLP, self).__init__()

        self.classifier = nn.Sequential(
            # Layer 1
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),

            # Layer 2
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate),

            # Output layer
            nn.Linear(hidden2, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : [N, input_dim] — pixel feature vectors

        Returns:
            logits : [N, num_classes]
        """
        return self.classifier(x)


# =============================================================================
# SNIPPET 9: Model Initialization
# =============================================================================

class PixelNet(nn.Module):
    """
    Full PixelNet model combining:
        1. VGG16 Backbone     — multi-scale feature extraction
        2. HypercolumnBuilder — feature map fusion
        3. PixelMLP           — per-pixel classification

    Training mode : Uses pixel sampling (sparse prediction)
    Inference mode: Dense prediction over full image
    """

    def __init__(
        self,
        pretrained_backbone: bool = True,
        num_sampled_pixels: int = NUM_SAMPLED_PIXELS
    ):
        super(PixelNet, self).__init__()

        self.backbone       = VGG16Backbone(pretrained=pretrained_backbone)
        self.hypercolumn    = HypercolumnBuilder()
        self.mlp            = PixelMLP()
        self.num_pixels     = num_sampled_pixels

    def forward(self, images: torch.Tensor, masks: torch.Tensor = None):
        """
        Args:
            images : [B, 3, H, W] — input batch
            masks  : [B, H, W]    — ground truth labels (training only)

        Returns:
            Training   : (logits [B*N, 2], labels [B*N])
            Inference  : logits  [B, 2, H, W]
        """
        B, C, H, W = images.shape

        # Step 1: Extract multi-scale features via VGG16
        feature_maps = self.backbone(images)

        # Step 2: Build hypercolumn [B, 960, H, W]
        hc = self.hypercolumn(feature_maps, target_size=(H, W))

        if self.training and masks is not None:
            # --- Sparse Training Path ---
            # Sample pixels and get their features + labels
            sampled_feat, sampled_labels = sample_pixels(hc, masks, self.num_pixels)
            sampled_feat   = sampled_feat.to(device)
            sampled_labels = sampled_labels.to(device)

            # Classify sampled pixels
            logits = self.mlp(sampled_feat)  # [B*N, 2]
            return logits, sampled_labels

        else:
            # --- Dense Inference Path ---
            # Reshape: [B, 960, H, W] → [B*H*W, 960]
            B, Chc, H, W = hc.shape
            hc_flat = hc.permute(0, 2, 3, 1).contiguous().view(-1, Chc)

            logits_flat = self.mlp(hc_flat)                    # [B*H*W, 2]
            logits_map  = logits_flat.view(B, H, W, 2)         # [B, H, W, 2]
            logits_map  = logits_map.permute(0, 3, 1, 2)       # [B, 2, H, W]

            return logits_map


def build_model(pretrained: bool = True) -> PixelNet:
    """Instantiate and move PixelNet to the target device."""
    model = PixelNet(pretrained_backbone=pretrained).to(device)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model initialized — Total params: {total_params:,} | Trainable: {trainable_params:,}")
    return model


# =============================================================================
# SNIPPET 10: Training Loop
# =============================================================================

def compute_iou(pred_mask: torch.Tensor, true_mask: torch.Tensor, num_classes: int = 2) -> float:
    """
    Compute mean Intersection over Union (mIoU) for semantic segmentation.

    Args:
        pred_mask  : [H, W] — predicted class map
        true_mask  : [H, W] — ground truth class map
        num_classes: Number of classes

    Returns:
        Mean IoU across all classes
    """
    iou_list = []
    for cls in range(num_classes):
        pred_cls = (pred_mask == cls)
        true_cls = (true_mask == cls)
        intersection = (pred_cls & true_cls).sum().item()
        union        = (pred_cls | true_cls).sum().item()
        if union == 0:
            continue
        iou_list.append(intersection / union)
    return float(np.mean(iou_list)) if iou_list else 0.0


def train_one_epoch(
    model: PixelNet,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    epoch: int
) -> dict:
    """Train model for one epoch and return average loss and accuracy."""
    model.train()

    epoch_loss = 0.0
    epoch_acc  = 0.0
    num_batches = len(loader)

    # tqdm progress bar for batch-level visualization
    progress = tqdm(loader, desc=f"Epoch {epoch:02d} [Train]", leave=False)

    for images, masks in progress:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Forward pass (sparse — returns sampled logits + labels)
        logits, labels = model(images, masks)

        # Compute cross-entropy loss on sampled pixels
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        # Accuracy on sampled pixels
        preds = logits.argmax(dim=1)
        acc   = (preds == labels).float().mean().item()

        epoch_loss += loss.item()
        epoch_acc  += acc

        progress.set_postfix(loss=f"{loss.item():.4f}", acc=f"{acc:.4f}")

    return {
        "loss": epoch_loss / num_batches,
        "acc":  epoch_acc  / num_batches,
    }


@torch.no_grad()
def validate_one_epoch(
    model: PixelNet,
    loader: DataLoader,
    criterion: nn.Module,
    epoch: int
) -> dict:
    """Validate model using dense prediction (inference mode)."""
    model.eval()

    epoch_loss = 0.0
    epoch_miou = 0.0
    num_batches = len(loader)

    progress = tqdm(loader, desc=f"Epoch {epoch:02d} [Val]  ", leave=False)

    for images, masks in progress:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        # Dense inference
        logits_map = model(images)              # [B, 2, H, W]

        # Compute loss over all pixels for validation
        B, C, H, W = logits_map.shape
        logits_flat = logits_map.permute(0, 2, 3, 1).contiguous().view(-1, C)
        labels_flat = masks.view(-1)
        loss = criterion(logits_flat, labels_flat)

        # mIoU on predicted segmentation map
        pred_masks = logits_map.argmax(dim=1)   # [B, H, W]
        batch_miou = np.mean([
            compute_iou(pred_masks[b].cpu(), masks[b].cpu())
            for b in range(B)
        ])

        epoch_loss += loss.item()
        epoch_miou += batch_miou

        progress.set_postfix(loss=f"{loss.item():.4f}", mIoU=f"{batch_miou:.4f}")

    return {
        "loss": epoch_loss / num_batches,
        "mIoU": epoch_miou / num_batches,
    }


def train(
    model: PixelNet,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
    lr: float = LEARNING_RATE,
    save_path: str = "pixelnet_best.pth"
) -> dict:
    """
    Full training loop with:
    - CrossEntropyLoss (handles class imbalance via weights)
    - Adam optimizer
    - Learning rate scheduler (ReduceLROnPlateau)
    - Best model checkpointing
    - tqdm epoch + batch progress bars

    Returns:
        history dict with train/val losses, accuracies, and mIoU
    """
    # Class weights to address fire/smoke imbalance
    class_weights = torch.tensor([0.3, 0.7], device=device)
    criterion  = nn.CrossEntropyLoss(weight=class_weights)
    optimizer  = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-5
    )
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, verbose=True
    )

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_mIoU": []}
    best_miou = 0.0

    # Outer epoch progress bar
    epoch_bar = tqdm(range(1, num_epochs + 1), desc="Training Progress")

    for epoch in epoch_bar:
        # ----- Train -----
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, epoch)

        # ----- Validate -----
        val_metrics = validate_one_epoch(model, val_loader, criterion, epoch)

        # Update LR scheduler based on validation loss
        scheduler.step(val_metrics["loss"])

        # Record history
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["acc"])
        history["val_loss"].append(val_metrics["loss"])
        history["val_mIoU"].append(val_metrics["mIoU"])

        # Save best model checkpoint
        if val_metrics["mIoU"] > best_miou:
            best_miou = val_metrics["mIoU"]
            torch.save(model.state_dict(), save_path)

        epoch_bar.set_postfix(
            train_loss=f"{train_metrics['loss']:.4f}",
            val_mIoU=f"{val_metrics['mIoU']:.4f}",
            best_mIoU=f"{best_miou:.4f}"
        )

    print(f"\nTraining complete. Best Val mIoU: {best_miou:.4f} — saved to {save_path}")
    return history


def plot_training_history(history: dict):
    """Plot training and validation loss + mIoU curves."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("PixelNet Training History", fontsize=15, fontweight="bold")

    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss", markersize=4)
    axes[0].plot(epochs, history["val_loss"],   "r-o", label="Val Loss",   markersize=4)
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history["train_acc"],  "b-o", label="Train Acc",  markersize=4)
    axes[1].plot(epochs, history["val_mIoU"],   "g-o", label="Val mIoU",   markersize=4)
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
    axes[1].set_title("Accuracy / mIoU"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Training history plot saved.")


# =============================================================================
# SNIPPET 11: Dense Prediction (Inference)
# =============================================================================

@torch.no_grad()
def predict_image(
    model: PixelNet,
    image_tensor: torch.Tensor
) -> np.ndarray:
    """
    Run dense segmentation inference on a single preprocessed image.

    Args:
        model        : Trained PixelNet (in eval mode)
        image_tensor : [3, H, W] — normalized image tensor

    Returns:
        pred_mask : [H, W] numpy array with class labels {0, 1}
    """
    model.eval()

    # Add batch dimension: [1, 3, H, W]
    image_batch = image_tensor.unsqueeze(0).to(device)

    # Dense forward pass (no mask passed → inference mode)
    logits_map = model(image_batch)              # [1, 2, H, W]

    # Argmax across class dimension → predicted class per pixel
    pred_mask = logits_map.squeeze(0).argmax(dim=0)  # [H, W]

    return pred_mask.cpu().numpy().astype(np.uint8)


@torch.no_grad()
def predict_batch(
    model: PixelNet,
    image_tensors: torch.Tensor
) -> np.ndarray:
    """
    Run dense inference on a batch of images.

    Args:
        model         : Trained PixelNet
        image_tensors : [B, 3, H, W]

    Returns:
        pred_masks : [B, H, W] numpy array
    """
    model.eval()
    image_tensors = image_tensors.to(device)
    logits_map    = model(image_tensors)          # [B, 2, H, W]
    pred_masks    = logits_map.argmax(dim=1)      # [B, H, W]
    return pred_masks.cpu().numpy().astype(np.uint8)


# =============================================================================
# SNIPPET 12: Visualization
# =============================================================================

# ImageNet denormalization stats (to reverse preprocessing for display)
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def denormalize(tensor: torch.Tensor) -> np.ndarray:
    """
    Reverse ImageNet normalization for visualization.

    Args:
        tensor : [3, H, W] — normalized image tensor

    Returns:
        np.ndarray [H, W, 3] uint8 RGB image
    """
    img = tensor.cpu().clone() * _STD + _MEAN
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)


def visualize_prediction(
    image_tensor: torch.Tensor,
    true_mask: np.ndarray,
    pred_mask: np.ndarray,
    title: str = "PixelNet Segmentation"
):
    """
    Display a 3-panel visualization: Input | Ground Truth | Prediction.

    Args:
        image_tensor : [3, H, W] — normalized input image
        true_mask    : [H, W]   — ground truth binary mask
        pred_mask    : [H, W]   — predicted binary mask
        title        : Figure title
    """
    image_rgb = denormalize(image_tensor)

    # Colorize masks: background=dark, fire/smoke=orange-red
    def colorize_mask(mask: np.ndarray) -> np.ndarray:
        colored = np.zeros((*mask.shape, 3), dtype=np.uint8)
        colored[mask == 0] = [30,  30,  30]    # background = near-black
        colored[mask == 1] = [255, 80,  10]    # fire/smoke = orange-red
        return colored

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    axes[0].imshow(image_rgb)
    axes[0].set_title("Input Image", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(colorize_mask(true_mask))
    axes[1].set_title("Ground Truth Mask", fontsize=11)
    axes[1].axis("off")

    axes[2].imshow(colorize_mask(pred_mask))
    axes[2].set_title("Predicted Mask", fontsize=11)
    axes[2].axis("off")

    # Legend
    legend_patches = [
        mpatches.Patch(color=(30/255, 30/255, 30/255), label="Background"),
        mpatches.Patch(color=(255/255, 80/255, 10/255), label="Fire / Smoke"),
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=2,
               fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, -0.05))

    plt.tight_layout()
    plt.savefig("prediction_visualization.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Visualization saved to prediction_visualization.png")


def visualize_overlay(
    image_tensor: torch.Tensor,
    pred_mask: np.ndarray,
    alpha: float = 0.45
):
    """
    Overlay the predicted segmentation mask on the original image.
    Fire/smoke regions are highlighted in red-orange.

    Args:
        image_tensor : [3, H, W] normalized image
        pred_mask    : [H, W] predicted mask {0, 1}
        alpha        : Overlay transparency (0 = image only, 1 = mask only)
    """
    image_rgb = denormalize(image_tensor).copy()

    # Apply colored overlay where fire/smoke detected
    overlay = image_rgb.copy()
    fire_pixels = pred_mask == 1
    overlay[fire_pixels] = [255, 60, 0]                    # Bright orange-red

    # Blend
    blended = (image_rgb * (1 - alpha) + overlay * alpha).astype(np.uint8)

    plt.figure(figsize=(8, 5))
    plt.imshow(blended)
    plt.title("Segmentation Overlay — Fire/Smoke Detected", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig("overlay_visualization.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Overlay saved to overlay_visualization.png")


# =============================================================================
# SNIPPET 13: Example Prediction (End-to-End Demo)
# =============================================================================

def run_demo_prediction(
    model: PixelNet,
    dataset: FireSmokeDataset,
    sample_index: int = 0
):
    """
    Run the full pipeline on a single sample from the dataset and visualize output.

    Steps:
        1. Load sample (image + ground truth mask) from dataset
        2. Run dense inference through trained PixelNet
        3. Compute mIoU for the sample
        4. Visualize: input | ground truth | prediction
        5. Show overlay

    Args:
        model        : Trained PixelNet (checkpoint loaded)
        dataset      : FireSmokeDataset instance
        sample_index : Index of the sample to run
    """
    model.eval()

    # Load sample from dataset
    image_tensor, true_mask_tensor = dataset[sample_index]
    true_mask = true_mask_tensor.numpy()

    # Run prediction
    print(f"Running inference on sample index {sample_index}...")
    pred_mask = predict_image(model, image_tensor)

    # Compute sample mIoU
    sample_miou = compute_iou(
        torch.from_numpy(pred_mask),
        torch.from_numpy(true_mask)
    )
    fire_pixels_gt   = (true_mask == 1).sum()
    fire_pixels_pred = (pred_mask == 1).sum()

    print(f"Sample mIoU    : {sample_miou:.4f}")
    print(f"GT fire pixels : {fire_pixels_gt}")
    print(f"Pred fire pxls : {fire_pixels_pred}")

    # Full visualization
    visualize_prediction(image_tensor, true_mask, pred_mask,
                         title=f"PixelNet Prediction — Sample #{sample_index} (mIoU: {sample_miou:.3f})")

    # Overlay visualization
    visualize_overlay(image_tensor, pred_mask)

    return pred_mask, sample_miou


def run_demo_on_raw_image(
    model: PixelNet,
    image_path: str,
    image_size: tuple = IMAGE_SIZE
):
    """
    Run PixelNet on any raw image file (no mask required).
    Useful for real-world fire hazard detection.

    Args:
        model      : Trained PixelNet
        image_path : Path to input image
        image_size : Resize target (must match training size)
    """
    model.eval()

    # Load and preprocess image
    image_pil = Image.open(image_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=FireSmokeDataset.MEAN, std=FireSmokeDataset.STD),
    ])
    image_tensor = transform(image_pil)

    # Predict
    pred_mask = predict_image(model, image_tensor)

    # Display overlay
    visualize_overlay(image_tensor, pred_mask)

    fire_coverage = (pred_mask == 1).mean() * 100
    print(f"Fire/Smoke coverage: {fire_coverage:.2f}% of image area")

    return pred_mask


# =============================================================================
# MAIN: Full Pipeline (uncomment and run in Colab)
# =============================================================================

if __name__ == "__main__":

    # ---- Step 1: Configure Kaggle and download dataset ----
    # from google.colab import files
    # uploaded = files.upload()                          # Upload kaggle.json
    # setup_kaggle_credentials("kaggle.json")
    # extract_dir = download_and_extract_dataset(
    #     dataset_slug="diversisai/fire-and-smoke",
    # )
    # images_dir, masks_dir = organize_dataset(extract_dir)

    # ---- Step 2: Build DataLoaders ----
    # train_loader, val_loader = build_dataloaders(
    #     images_dir=images_dir,
    #     masks_dir=masks_dir,
    #     batch_size=BATCH_SIZE
    # )

    # ---- Step 3: Initialize model ----
    # model = build_model(pretrained=True)

    # ---- Step 4: Train ----
    # history = train(
    #     model=model,
    #     train_loader=train_loader,
    #     val_loader=val_loader,
    #     num_epochs=NUM_EPOCHS,
    #     lr=LEARNING_RATE,
    #     save_path="pixelnet_best.pth"
    # )
    # plot_training_history(history)

    # ---- Step 5: Load best checkpoint and run demo ----
    # model.load_state_dict(torch.load("pixelnet_best.pth", map_location=device))

    # full_dataset = FireSmokeDataset(images_dir, masks_dir, IMAGE_SIZE)
    # pred_mask, miou = run_demo_prediction(model, full_dataset, sample_index=0)

    # ---- Step 6 (Optional): Predict on a raw image ----
    # run_demo_on_raw_image(model, "path/to/fire_image.jpg")

    print("PixelNet pipeline script loaded successfully.")
    print("Uncomment the sections above to run the full training pipeline in Colab.")